# KataGo Remote Engine on Google Colab

Run KataGo in a Colab runtime and connect to it from KaTrain or SWHub using a temporary WebSocket URL.

## Quick Start

1. Open this notebook in Google Colab.
2. Choose `Runtime -> Change runtime type -> T4 GPU` (or the available NVIDIA GPU runtime). This notebook has no CPU or OpenCL mode.
3. In the setup cell below, edit only the `USER SETTINGS` constants near the top if needed. The defaults are fine for normal use.
4. Run the setup cell and wait for `Final WebSocket URL`. Startup can take several minutes the first time, especially on GPU.
5. Copy the printed `wss://.../katago` URL.
6. In KaTrain or SWHub, open KataGo settings, choose the remote engine option, and paste the WebSocket URL.
7. Open the printed monitor URL, usually `https://.../monitor`, to watch live status, logs, tuning, queries, and errors.

## Recommended Defaults

| Setting | Default | When to change it |
| --- | --- | --- |
| KataGo backend | CUDA + cuDNN | Fixed. A usable NVIDIA GPU runtime is required; T4 is the intended free-tier target. |
| `TUNNEL_PROVIDER` | `"auto"` | Use `"pinggy"` or `"localhostrun"` if Cloudflare is blocked or unreliable. |
| `MAIN_MODEL_PRESET` | `"transformer_medium"` | Recommended strength/speed default for remote GPU analysis. Use an older convolutional preset only for comparison. |
| `HUMAN_MODEL_PRESET` | `"official_human_sl_v0"` | Use `"disabled"` if you do not need human/rank-style features. |
| `STOP_OLD_PROCESSES` | `True` | Keep this on when rerunning the setup cell in the same Colab runtime. |

## Tunnel Fallbacks

`TUNNEL_PROVIDER = "auto"` tries providers in `TUNNEL_FALLBACK_ORDER`:

1. `cloudflare`: best default. No account, random `trycloudflare.com` URL, usually the cleanest `wss://.../katago` connection. Temporary and occasionally flaky.
2. `pinggy`: first fallback. Uses SSH to `free.pinggy.io`, no download or signup, HTTPS/WebSocket forwarding, free tunnel timeout is currently about 60 minutes.
3. `localhostrun`: second fallback. Uses SSH to `localhost.run`, no download or signup, free HTTPS URL. Free domains change regularly and may be speed-limited.

To prefer a fallback automatically, change the order, for example:

```python
TUNNEL_FALLBACK_ORDER = ["pinggy", "cloudflare", "localhostrun"]
```

Use `TUNNEL_PROVIDER = "none"` only for testing inside Colab. Your local KaTrain/SWHub usually cannot reach Colab runtime `127.0.0.1`.

## Model Choices

Main model presets:

- `transformer_small`: compact transformer; stronger per visit than b18 and usually as fast or faster.
- `transformer_medium`: recommended default; stronger than b28 per visit with a moderate download/runtime cost.
- `transformer_large`: strongest transformer preset; use only when the extra download/runtime cost is acceptable.
- `colab_fast_b18`: older convolutional fallback for comparison.
- `latest_b28`: stronger current official KataGo network, slower to download/tune/run.
- `strongest_b40_zhizi`: strongest preset here, but heavy. Use only with a good GPU runtime.
- `custom`: paste your own `.bin.gz` URL into `MODEL_URL` and its SHA-256 into `MODEL_SHA256` (or leave the hash blank only if the publisher provides none).

Human SL model presets:

- `official_human_sl_v0`: official Human SL model for human/rank-style features.
- `disabled`: skip Human SL download and startup.
- `custom`: paste your own Human SL model URL and SHA-256.

Human SL is not a replacement for the main model in normal use. It is loaded as an extra `-human-model` so KaTrain/SWHub can request `humanSLProfile` behavior.

KataGo binaries and models are stored only in the current Colab runtime under `/content`. Colab runtimes are temporary, so a fresh runtime downloads them again. This notebook does not mount or access Google Drive. Colab's no-cost tier does not guarantee GPU access, a T4 specifically, or uninterrupted runtime duration. Its current policy may also terminate tunneled remote-control or primarily web-service workloads; use a paid positive compute-unit balance when required.

## Troubleshooting

- If the URL stops working, the Colab runtime or tunnel probably disconnected. Rerun the setup cell.
- A fresh Colab runtime downloads KataGo and the selected models again because runtime storage is temporary.
- If Cloudflare does not produce a URL, set `TUNNEL_PROVIDER = "pinggy"` and rerun.
- CUDA uses the KataGo v1.18.0 cuDNN 9.8 build when Colab exposes cuDNN 9, and the compatible cuDNN 8.9.7 build otherwise. The cuDNN 8 fallback works but is much slower for transformer models.
- If no NVIDIA GPU is assigned, or CUDA startup fails, change the Colab runtime to a GPU and rerun. The notebook stops instead of falling back to OpenCL or CPU.
- If KaTrain does not connect after pasting the URL, fully close and reopen KaTrain. In SWHub, click the status dot to reconnect.
- Before closing the runtime, run the optional stop cell at the bottom to clean up server and tunnel processes.



In [ ]:
# @title Start KataGo Remote Engine - edit user settings below


# CUDA-only runtime. Select a Colab T4 GPU (or another available NVIDIA GPU).
# Setup stops with a clear error if Colab did not attach a usable GPU.

# If True, old server/tunnel/KataGo processes are stopped before starting.
STOP_OLD_PROCESSES = True

# Tunnel provider:
#   "auto"         -> try TUNNEL_FALLBACK_ORDER in order
#   "cloudflare"   -> Cloudflare Quick Tunnel, no account, random trycloudflare URL
#   "pinggy"       -> Pinggy free SSH tunnel, no download, 60-minute free timeout
#   "localhostrun" -> localhost.run free SSH tunnel, no download, changing free domains
#   "none"         -> skip public tunnel; print local ws/http URLs only
TUNNEL_PROVIDER = "auto"
TUNNEL_FALLBACK_ORDER = ["cloudflare", "pinggy", "localhostrun"]

# Tunnel startup timeouts.
CLOUDFLARE_TIMEOUT_SECONDS = 120
PINGGY_TIMEOUT_SECONDS = 90
LOCALHOSTRUN_TIMEOUT_SECONDS = 90

# SSH tunnel options for Pinggy/localhost.run fallbacks.
SSH_SERVER_ALIVE_INTERVAL_SECONDS = 30
SSH_CONNECT_TIMEOUT_SECONDS = 15
PINGGY_DEBUG_PORT = 4300

# If True, re-download KataGo even if the folder already exists.
FORCE_REDOWNLOAD_KATAGO = False

# If True, re-download model even if model.bin.gz already exists.
FORCE_REDOWNLOAD_MODEL = False

# If True, re-download Human SL model even if it already exists.
FORCE_REDOWNLOAD_HUMAN_MODEL = False

# Backward-compatible aliases for older pasted copies of the settings block.
FORCE_REDOWLOAD_KATAGO = FORCE_REDOWNLOAD_KATAGO
FORCE_REDOWLOAD_MODEL = FORCE_REDOWNLOAD_MODEL
FORCE_REDOWLOAD_HUMAN_MODEL = FORCE_REDOWNLOAD_HUMAN_MODEL

# KataGo 1.18.0 release URLs. Transformer models require 1.17 or newer.
KATAGO_RELEASE_CACHE_KEY = "v1_18_0"
GPU_CUDA_CUDNN9_KATAGO_URL = "https://github.com/lightvector/KataGo/releases/download/v1.18.0/katago-v1.18.0-cuda12.1-cudnn9.8.0-linux-x64.zip"
GPU_CUDA_CUDNN9_KATAGO_SHA256 = "a152b3218f0f7f7bfc80d3e0522b659f7464cfe53c35948fcd3268abcd418ebf"
GPU_CUDA_CUDNN8_KATAGO_URL = "https://github.com/lightvector/KataGo/releases/download/v1.18.0/katago-v1.18.0-cuda12.1-cudnn8.9.7-linux-x64.zip"
GPU_CUDA_CUDNN8_KATAGO_SHA256 = "88a5a424c8cab9ee5a35c475a6c22633a9f5a43ea959878cb786c6f73955a7d6"

# Main analysis model. The medium transformer is the recommended default.
MAIN_MODEL_PRESET = "transformer_medium"
MODEL_URL = "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz"
MODEL_FILENAME = "b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz"
MODEL_SHA256 = "c04db4a503721d948bb720324f3cbdac6088cc9eb243632f020e4b6846f58995"

# Human SL model support.
# Keep this True if you use KaTrain/SWHub human-like/rank/policy features.
# KataGo needs this extra model when KaTrain sends overrideSettings.humanSLProfile.
ENABLE_HUMAN_MODEL = True
HUMAN_MODEL_PRESET = "official_human_sl_v0"
HUMAN_MODEL_URL = "https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz"
HUMAN_MODEL_FILENAME = "b18c384nbt-humanv0.bin.gz"
HUMAN_MODEL_SHA256 = "637746e44f0efe00ad1245a50aa9bbf0716efe364c43965ead97bd6835d84ab5"

# Local self-test profile for Human SL. KaTrain can still choose other profiles.
# Valid examples include rank_20k...rank_9d, preaz_20k...preaz_9d, proyear_1800...proyear_2023.
HUMAN_MODEL_TEST_PROFILE = "rank_4k"

# GPU config defaults. KaTrain/SWHub can override visits per query.
GPU_MAX_VISITS = 500
GPU_SEARCH_THREADS = 4
GPU_ANALYSIS_THREADS = 4
GPU_NN_CACHE_POWER = 21
GPU_NN_MAX_BATCH_SIZE = 16
GPU_CUDA_DEVICE = 0

# Local server port inside Colab.
SERVER_PORT = 8000

# WebSocket route used by KaTrain.
KATAGO_WS_PATH = "/katago"

# Large KataGo JSON output buffer.
SUBPROCESS_BUFFER_MB = 50

# How long to wait for KataGo startup.
# First GPU run with Human SL can tune two models and take several minutes.
KATAGO_STARTUP_TIMEOUT_SECONDS = 900

# Print latest server log every N seconds while waiting for startup.
STARTUP_LOG_INTERVAL_SECONDS = 15
STARTUP_LOG_TAIL_CHARS = 2200

# Monitor log memory.
MONITOR_LOG_LIMIT = 1200

# ============================================================
# Script starts here. Usually do not edit below this line.
# ============================================================

import os
import ctypes
import hashlib
import re
import json
import time
import signal
import shutil
import subprocess
import urllib.request
from pathlib import Path

RUNTIME_BASE = Path("/content")
BASE = RUNTIME_BASE
BASE.mkdir(parents=True, exist_ok=True)
CLOUDFLARED_PATH = RUNTIME_BASE / "cloudflared"
CLOUDFLARED_VERSION = "2026.8.2"
CLOUDFLARED_URL = f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
CLOUDFLARED_SHA256 = "fcfb02b575a52ca1af2e3267af4e1517bcdeb30ac48c834c69abaed3c0576ad2"
CLOUDFLARE_LOG_PATH = RUNTIME_BASE / "cloudflared.log"
PINGGY_LOG_PATH = RUNTIME_BASE / "pinggy_tunnel.log"
LOCALHOSTRUN_LOG_PATH = RUNTIME_BASE / "localhostrun_tunnel.log"
WS_SERVER_PATH = RUNTIME_BASE / "katago_ws_server.py"
WS_LOG_PATH = RUNTIME_BASE / "katago_ws_server.log"

MAIN_MODEL_PRESETS = {
    "transformer_small": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c384h6nbttflrs.bin.gz",
        "filename": "b10c384h6nbttflrs.bin.gz",
        "sha256": "0ba27eced5180b3e3d0b898b280c541112989765e789d1eb6cd0d31b2b2c1229",
        "note": "Recommended compact transformer; stronger per visit than b18 and usually as fast or faster.",
    },
    "transformer_medium": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz",
        "filename": "b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz",
        "sha256": "c04db4a503721d948bb720324f3cbdac6088cc9eb243632f020e4b6846f58995",
        "note": "Medium transformer; stronger per visit than b28 and usually as fast or faster.",
    },
    "transformer_large": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b11c768h12nbt3tflrs-fson-silu.bin.gz",
        "filename": "b11c768h12nbt3tflrs-fson-silu.bin.gz",
        "sha256": "1881600caab9e9d85a3dd6a019e9b8e7d2c237b5f984e13ed49a8645be3077c6",
        "note": "Largest transformer; stronger than b40 Zhizi and intended for capable runtimes.",
    },
    "colab_fast_b18": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b18c384nbt-s9996604416-d4316597426.bin.gz",
        "filename": "kata1-b18c384nbt-s9996604416-d4316597426.bin.gz",
        "sha256": "9d7a6afed8ff5b74894727e156f04f0cd36060a24824892008fbb6e0cba51f1d",
        "note": "Fast lightweight b18 model, good for free Colab.",
    },
    "latest_b28": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b28c512nbt-s13255194368-d5935380940.bin.gz",
        "filename": "kata1-b28c512nbt-s13255194368-d5935380940.bin.gz",
        "sha256": "c5bca453d7b08ea8df6546439325d4dd681e77d975e1da3f7593d771147b73bc",
        "note": "Latest official kata1 b28 network as of 2026-06-07; stronger but slower.",
    },
    "strongest_b40_zhizi": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-zhizi-b40c768nbt-s11272M-d5935M.bin.gz",
        "filename": "kata1-zhizi-b40c768nbt-s11272M-d5935M.bin.gz",
        "sha256": "15fb3baf85cdb6578e6c19b65e6201ca906cec3ba5fee19039d05221d57eb0e8",
        "note": "Strongest confidently-rated network on katagotraining; heavy for Colab.",
    },
}

HUMAN_MODEL_PRESETS = {
    "official_human_sl_v0": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz",
        "filename": "b18c384nbt-humanv0.bin.gz",
        "sha256": "637746e44f0efe00ad1245a50aa9bbf0716efe364c43965ead97bd6835d84ab5",
        "note": "Official Human SL model for humanSLProfile/rank-like analysis.",
    },
}


def apply_model_presets():
    global MODEL_URL, MODEL_FILENAME, MODEL_SHA256
    global HUMAN_MODEL_URL, HUMAN_MODEL_FILENAME, HUMAN_MODEL_SHA256
    global ENABLE_HUMAN_MODEL

    main_preset = MAIN_MODEL_PRESET.lower().strip()
    if main_preset != "custom":
        if main_preset not in MAIN_MODEL_PRESETS:
            choices = sorted([*MAIN_MODEL_PRESETS.keys(), "custom"])
            raise ValueError(f"MAIN_MODEL_PRESET must be one of {choices}")
        data = MAIN_MODEL_PRESETS[main_preset]
        MODEL_URL = data["url"]
        MODEL_FILENAME = data["filename"]
        MODEL_SHA256 = data["sha256"]

    human_preset = HUMAN_MODEL_PRESET.lower().strip()
    if human_preset == "disabled":
        ENABLE_HUMAN_MODEL = False
    elif human_preset != "custom":
        if human_preset not in HUMAN_MODEL_PRESETS:
            choices = sorted([*HUMAN_MODEL_PRESETS.keys(), "custom", "disabled"])
            raise ValueError(f"HUMAN_MODEL_PRESET must be one of {choices}")
        data = HUMAN_MODEL_PRESETS[human_preset]
        HUMAN_MODEL_URL = data["url"]
        HUMAN_MODEL_FILENAME = data["filename"]
        HUMAN_MODEL_SHA256 = data["sha256"]


def run(cmd, check=True):
    print(f"\n$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout


def command_output(cmd):
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return result.returncode, result.stdout


def detected_gpu_name():
    code, out = command_output("nvidia-smi --query-gpu=name --format=csv,noheader")
    if code != 0 or not out.strip():
        return None
    return out.strip().splitlines()[0].strip()


def decode_cudnn_version(value):
    if value is None:
        return None
    value = int(value)
    if value >= 10000:
        return value // 10000, (value % 10000) // 100, value % 100
    return value // 1000, (value % 1000) // 100, value % 100


def detected_cudnn_version():
    try:
        import torch
        version = decode_cudnn_version(torch.backends.cudnn.version())
        if version is not None:
            return version
    except Exception:
        pass
    for library_name in ("libcudnn.so.9", "libcudnn.so.8"):
        try:
            library = ctypes.CDLL(library_name)
            library.cudnnGetVersion.restype = ctypes.c_size_t
            version = decode_cudnn_version(library.cudnnGetVersion())
            if version is not None:
                return version
        except Exception:
            pass
    return None


def choose_cuda_runtime():
    gpu_name = detected_gpu_name()
    if not gpu_name:
        raise RuntimeError(
            "This notebook requires a Colab NVIDIA GPU runtime. "
            "Choose Runtime -> Change runtime type -> T4 GPU and rerun."
        )
    cudnn_version = detected_cudnn_version()
    if cudnn_version is None:
        raise RuntimeError(
            "A compatible cuDNN runtime was not detected. "
            "Reconnect to a fresh Colab GPU runtime and rerun."
        )
    use_cudnn9 = cudnn_version >= (9, 8, 0)
    use_cudnn8 = (8, 9, 7) <= cudnn_version < (9, 0, 0)
    if not use_cudnn9 and not use_cudnn8:
        detected = ".".join(str(part) for part in cudnn_version)
        raise RuntimeError(
            f"Detected cuDNN {detected}; KataGo v1.18 requires "
            "cuDNN >= 9.8.0 or the compatibility cuDNN 8.9.7 runtime."
        )
    cudnn_label = ".".join(str(part) for part in cudnn_version)
    return {
        "mode": "gpu",
        "backend": "cuda",
        "mode_name": f"{gpu_name} CUDA (cuDNN {cudnn_label})",
        "gpu_name": gpu_name,
        "katago_cache_dir": BASE / f"katago_{KATAGO_RELEASE_CACHE_KEY}_cuda_cudnn{9 if use_cudnn9 else 8}",
        "katago_dir": RUNTIME_BASE / f"katago_{KATAGO_RELEASE_CACHE_KEY}_cuda_cudnn{9 if use_cudnn9 else 8}",
        "katago_url": GPU_CUDA_CUDNN9_KATAGO_URL if use_cudnn9 else GPU_CUDA_CUDNN8_KATAGO_URL,
        "katago_sha256": GPU_CUDA_CUDNN9_KATAGO_SHA256 if use_cudnn9 else GPU_CUDA_CUDNN8_KATAGO_SHA256,
    }


CONFIG_DATA = choose_cuda_runtime()
apply_model_presets()
KATAGO_CACHE_DIR = CONFIG_DATA["katago_cache_dir"]
KATAGO_DIR = CONFIG_DATA["katago_dir"]
MODEL_DIR = BASE / "models"


def stop_old_processes():
    print("\n[1/12] Stopping old processes...")
    current_pid = os.getpid()

    try:
        out = subprocess.check_output(["ps", "-eo", "pid=,args="], text=True)
    except Exception as exc:
        print("Could not list processes:", exc)
        return

    kill_keywords = [
        "katago_ws_server.py",
        "cloudflared tunnel",
        "katago analysis",
        "free.pinggy.io",
        "localhost.run",
    ]

    for raw_line in out.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split(None, 1)
        if len(parts) < 2:
            continue
        if not parts[0].isdigit():
            continue

        pid = int(parts[0])
        cmd = parts[1]

        if pid == current_pid:
            continue

        if any(keyword in cmd for keyword in kill_keywords):
            print("Stopping:", pid, cmd)
            try:
                os.kill(pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            except PermissionError:
                print("No permission to stop:", pid)

    time.sleep(2)


def install_dependencies():
    print("\n[2/12] Installing dependencies...")
    run("apt update -y", check=False)

    run("apt install -y wget unzip p7zip-full libzip4 libomp-dev curl openssh-client", check=True)

    run("pip install fastapi==0.141.1 uvicorn==0.52.4 nest_asyncio==1.6.0 websockets==17.0.1 -q", check=True)


def print_system_info():
    print("\n[3/12] System info...")
    print("Selected mode:", CONFIG_DATA["mode_name"])
    print("Main model preset:", MAIN_MODEL_PRESET)
    if MAIN_MODEL_PRESET.lower().strip() in MAIN_MODEL_PRESETS:
        print("Main model note:", MAIN_MODEL_PRESETS[MAIN_MODEL_PRESET.lower().strip()]["note"])
    print("Main model URL:", MODEL_URL)
    print("Human model preset:", HUMAN_MODEL_PRESET)
    if ENABLE_HUMAN_MODEL and HUMAN_MODEL_PRESET.lower().strip() in HUMAN_MODEL_PRESETS:
        print("Human model note:", HUMAN_MODEL_PRESETS[HUMAN_MODEL_PRESET.lower().strip()]["note"])
    print("Human model enabled:", ENABLE_HUMAN_MODEL)
    print("Runtime storage folder:", BASE)
    print("KataGo cache folder:", KATAGO_CACHE_DIR)
    print("KataGo folder:", KATAGO_DIR)
    print("Model folder:", MODEL_DIR)
    print("Assigned NVIDIA GPU:", CONFIG_DATA["gpu_name"])
    run("nvidia-smi", check=False)
    run("lscpu | head -80", check=False)


def stage_katago_runtime():
    katago_bin = KATAGO_DIR / "katago"

    if KATAGO_CACHE_DIR == KATAGO_DIR:
        run(f"chmod +x {katago_bin}", check=False)
        return

    if KATAGO_DIR.exists():
        shutil.rmtree(KATAGO_DIR)

    print(f"Staging KataGo executable files into runnable runtime folder: {KATAGO_DIR}")
    shutil.copytree(
        KATAGO_CACHE_DIR,
        KATAGO_DIR,
        ignore=shutil.ignore_patterns("*.zip", "models", "logs"),
    )
    run(f"chmod +x {katago_bin}", check=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_download(path, expected_sha256=""):
    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(f"Download produced an empty file: {path}")
    if expected_sha256:
        actual = sha256_file(path)
        if actual.lower() != expected_sha256.lower():
            raise RuntimeError(f"SHA-256 mismatch for {path.name}: expected {expected_sha256}, got {actual}")


def download_file(url, path, expected_sha256="", force=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not force:
        try:
            verify_download(path, expected_sha256)
            print(f"Already exists and verified: {path}")
            return
        except Exception as exc:
            print(f"Cached file failed verification; downloading again: {exc}")
    tmp = path.with_name(path.name + ".tmp")
    tmp.unlink(missing_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "SWHub-KataGo-Colab/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=120) as response, open(tmp, "wb") as target:
            shutil.copyfileobj(response, target)
    except Exception as exc:
        print(f"Python download failed ({exc!r}); trying wget...")
        tmp.unlink(missing_ok=True)
        subprocess.run(["wget", "--tries=3", "--timeout=60", "-O", str(tmp), url], check=True)
    try:
        verify_download(tmp, expected_sha256)
        os.replace(tmp, path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


def download_katago():
    print("\n[4/12] Downloading KataGo...")
    KATAGO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(KATAGO_CACHE_DIR)

    cache_bin = KATAGO_CACHE_DIR / "katago"
    if FORCE_REDOWNLOAD_KATAGO and cache_bin.exists():
        cache_bin.unlink()

    if not cache_bin.exists():
        download_file(CONFIG_DATA["katago_url"], KATAGO_CACHE_DIR / "katago.zip", CONFIG_DATA["katago_sha256"], force=True)
        run("unzip -o katago.zip", check=True)
    else:
        print("KataGo already exists in cache. Skipping download.")

    stage_katago_runtime()
    katago_bin = KATAGO_DIR / "katago"
    code, version_out = command_output(f"{katago_bin} version")
    print(version_out, flush=True)
    if code != 0:
        raise RuntimeError(f"CUDA KataGo failed to run on {CONFIG_DATA['gpu_name']}:\n{version_out}")
    if not re.search(r"(?:^|[^0-9])1\.18\.0(?:[^0-9]|$)", version_out):
        raise RuntimeError(f"Expected KataGo 1.18.0, got:\n{version_out}")


def download_model():
    print("\n[5/12] Downloading model files...")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    model_path = MODEL_DIR / MODEL_FILENAME
    download_file(MODEL_URL, model_path, MODEL_SHA256, force=FORCE_REDOWNLOAD_MODEL)

    run(f'ls -lh "{model_path}"', check=True)

    if ENABLE_HUMAN_MODEL:
        human_model_path = MODEL_DIR / HUMAN_MODEL_FILENAME
        print("Checking Human SL model...")
        download_file(HUMAN_MODEL_URL, human_model_path, HUMAN_MODEL_SHA256, force=FORCE_REDOWNLOAD_HUMAN_MODEL)

        run(f'ls -lh "{human_model_path}"', check=True)
    else:
        print("Human SL model disabled by ENABLE_HUMAN_MODEL = False")


def write_analysis_config():
    print("\n[6/12] Writing analysis.cfg...")

    cfg = f"""logDir = {KATAGO_DIR}/logs

maxVisits = {GPU_MAX_VISITS}

numSearchThreadsPerAnalysisThread = {GPU_SEARCH_THREADS}
numAnalysisThreads = {GPU_ANALYSIS_THREADS}

nnCacheSizePowerOfTwo = {GPU_NN_CACHE_POWER}
nnMaxBatchSize = {GPU_NN_MAX_BATCH_SIZE}
numNNServerThreadsPerModel = 2

cudaDeviceToUseThread0 = {GPU_CUDA_DEVICE}
"""

    config_path = KATAGO_DIR / "analysis.cfg"
    config_path.write_text(cfg)
    print("Selected config mode:", CONFIG_DATA["mode_name"])
    print("Wrote:", config_path)
    print(cfg)


def write_ws_server():
    print("\n[7/12] Writing WebSocket server with live monitor...")

    server_template = r'''
import asyncio
import json
import time
import uuid
from collections import deque
from typing import Dict, Tuple

import nest_asyncio
import uvicorn
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Request
from fastapi.responses import HTMLResponse, PlainTextResponse, StreamingResponse

MODE_NAME = __MODE_NAME__
KATAGO_BIN = __KATAGO_BIN__
MODEL = __MODEL__
HUMAN_MODEL = __HUMAN_MODEL__
CONFIG = __CONFIG__
SERVER_PORT = __SERVER_PORT__
SUBPROCESS_LIMIT = __SUBPROCESS_LIMIT__
LOG_LIMIT = __LOG_LIMIT__

app = FastAPI()
katago_proc = None
pending: Dict[str, Tuple[WebSocket, asyncio.Lock]] = {}
pending_expected_turns: Dict[str, set] = {}
pending_completed_turns: Dict[str, set] = {}
katago_write_lock = asyncio.Lock()
LOGS = deque(maxlen=LOG_LIMIT)
EVENT_SUBSCRIBERS = set()
STALE_RESULT_IDS = set()
STALE_RESULT_ORDER = deque()

STATE = {
    "mode": MODE_NAME,
    "started_at": time.time(),
    "katago_started": False,
    "katago_ready": False,
    "websocket_clients_total": 0,
    "websocket_clients_active": 0,
    "queries_received": 0,
    "results_sent": 0,
    "errors": 0,
    "warnings": 0,
    "last_query_id": None,
    "last_result_id": None,
    "last_error": None,
    "last_best_move": None,
    "last_visits": None,
    "last_winrate": None,
    "last_score_lead": None,
    "human_model_enabled": bool(HUMAN_MODEL),
    "pending_queries": 0,
    "stale_results_dropped": 0,
}


def now_time():
    return time.strftime("%H:%M:%S")


def now_ts():
    return time.time()


async def emit(event_type, message, extra=None):
    if event_type == "error":
        STATE["errors"] += 1
    if event_type == "warning":
        STATE["warnings"] += 1

    STATE["pending_queries"] = len(pending)

    item = {
        "ts": now_ts(),
        "time": now_time(),
        "type": event_type,
        "message": str(message),
        "state": dict(STATE),
    }
    if extra:
        item.update(extra)

    LOGS.append(item)

    dead = []
    for queue in EVENT_SUBSCRIBERS:
        try:
            queue.put_nowait(item)
        except Exception:
            dead.append(queue)

    for queue in dead:
        EVENT_SUBSCRIBERS.discard(queue)

    print(f"[{item['time']}] {event_type}: {message}", flush=True)


def first_stale_result(query_id):
    if not query_id or query_id in STALE_RESULT_IDS:
        return False
    STALE_RESULT_IDS.add(query_id)
    STALE_RESULT_ORDER.append(query_id)
    while len(STALE_RESULT_ORDER) > 500:
        STALE_RESULT_IDS.discard(STALE_RESULT_ORDER.popleft())
    return True


async def write_to_katago(payload):
    if katago_proc is None or katago_proc.stdin is None:
        raise RuntimeError("KataGo process is not running.")
    if katago_proc.returncode is not None:
        raise RuntimeError(f"KataGo process already exited with code {katago_proc.returncode}.")

    async with katago_write_lock:
        katago_proc.stdin.write((json.dumps(payload) + "\n").encode("utf-8"))
        await katago_proc.stdin.drain()


def finish_pending_query(query_id):
    pending.pop(query_id, None)
    pending_expected_turns.pop(query_id, None)
    pending_completed_turns.pop(query_id, None)
    STATE["pending_queries"] = len(pending)


async def terminate_katago_query(query_id, reason):
    finish_pending_query(query_id)
    terminate = {"id": f"terminate-{query_id}-{uuid.uuid4()}", "action": "terminate", "terminateId": query_id}
    try:
        await write_to_katago(terminate)
        await emit("query", f"Terminated query id={query_id}: {reason}", {"query_id": query_id})
    except Exception as exc:
        STATE["last_error"] = repr(exc)
        await emit("error", f"Could not terminate query id={query_id}: {repr(exc)}")


async def terminate_pending_query(query_id, reason):
    if query_id not in pending:
        return
    await terminate_katago_query(query_id, reason)


async def start_katago():
    global katago_proc
    await emit("system", f"Starting KataGo mode={MODE_NAME}")

    katago_args = [KATAGO_BIN, "analysis", "-model", MODEL, "-config", CONFIG]

    if HUMAN_MODEL:
        katago_args.extend(["-human-model", HUMAN_MODEL])
        await emit("system", f"Human SL model enabled: {HUMAN_MODEL}")
    else:
        await emit("warning", "Human SL model disabled. KaTrain humanSLProfile queries will fail.")

    await emit("system", "Launch command: " + " ".join(katago_args))

    katago_proc = await asyncio.create_subprocess_exec(
        *katago_args,
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
        limit=SUBPROCESS_LIMIT,
    )

    STATE["katago_started"] = True
    asyncio.create_task(read_katago_stdout())
    asyncio.create_task(read_katago_stderr())
    await emit("system", "KataGo process started.")


async def read_katago_stdout():
    while True:
        try:
            line = await katago_proc.stdout.readline()
        except Exception as exc:
            STATE["last_error"] = repr(exc)
            await emit("error", f"KataGo stdout read error: {repr(exc)}")
            break

        if not line:
            await emit("system", "KataGo stdout closed.")
            break

        text = line.decode("utf-8", errors="ignore").strip()
        if not text:
            continue

        try:
            data = json.loads(text)
            query_id = data.get("id")
            terminate_id = data.get("terminateId")
            STATE["last_result_id"] = query_id

            if terminate_id:
                await emit("result", f"KataGo acknowledged termination id={terminate_id}", {"query_id": terminate_id})
                continue

            if query_id is None and (data.get("error") or data.get("warning") or data.get("noResults")):
                detail = data.get("error") or data.get("warning") or "KataGo returned no results"
                level = "error" if data.get("error") or data.get("noResults") else "warning"
                await emit(level, f"Unscoped KataGo {level}: {detail}")
                delivered = set()
                for websocket, send_lock in list(pending.values()):
                    if id(websocket) in delivered:
                        continue
                    delivered.add(id(websocket))
                    try:
                        async with send_lock:
                            await websocket.send_text(json.dumps(data))
                    except Exception as exc:
                        await emit("warning", f"Could not forward unscoped KataGo error: {exc!r}")
                continue

            target = pending.get(query_id)
            if not target:
                STATE["stale_results_dropped"] += 1
                if first_stale_result(query_id):
                    await emit("stale", f"Dropped stale result id={query_id}; client already cancelled or replaced this query")
                continue

            websocket, send_lock = target
            async with send_lock:
                await websocket.send_text(json.dumps(data))

            STATE["results_sent"] += 1
            move_infos = data.get("moveInfos", [])
            best_move = move_infos[0].get("move") if move_infos else None
            visits = move_infos[0].get("visits") if move_infos else None
            winrate = move_infos[0].get("winrate") if move_infos else None
            score_lead = move_infos[0].get("scoreLead") if move_infos else None

            STATE["last_best_move"] = best_move
            STATE["last_visits"] = visits
            STATE["last_winrate"] = winrate
            STATE["last_score_lead"] = score_lead

            await emit(
                "result",
                f"Sent result id={query_id}, best={best_move}, visits={visits}, winrate={winrate}, scoreLead={score_lead}",
                {"query_id": query_id, "best_move": best_move, "visits": visits, "winrate": winrate, "score_lead": score_lead},
            )

            if data.get("error") or data.get("noResults"):
                finish_pending_query(query_id)
            elif data.get("isDuringSearch") is False:
                expected = pending_expected_turns.get(query_id, set())
                if not expected:
                    finish_pending_query(query_id)
                else:
                    turn_number = data.get("turnNumber")
                    if turn_number is not None:
                        pending_completed_turns.setdefault(query_id, set()).add(int(turn_number))
                    if expected.issubset(pending_completed_turns.get(query_id, set())):
                        finish_pending_query(query_id)

        except Exception as exc:
            STATE["last_error"] = repr(exc)
            await emit("error", f"Error forwarding KataGo output: {repr(exc)}")


async def read_katago_stderr():
    while True:
        line = await katago_proc.stderr.readline()
        if not line:
            break

        msg = line.decode("utf-8", errors="ignore").rstrip()
        lower = msg.lower()

        if "started, ready to begin handling requests" in lower:
            STATE["katago_ready"] = True
            await emit("katago", "KataGo ready.")
        elif "using cuda backend" in lower or "loaded model" in lower:
            await emit("katago", msg)
        elif "performing autotuning" in lower or "done tuning" in lower or "saved results" in lower:
            await emit("tuning", msg)
        elif "tuning " in msg or "ErrorProp" in msg or "Calls/sec" in msg:
            await emit("tuning", msg)
        elif "warning" in lower:
            await emit("warning", msg)
        elif "error" in lower or "failed" in lower:
            STATE["last_error"] = msg
            await emit("error", msg)
        else:
            print("KataGo:", msg, flush=True)


@app.on_event("startup")
async def startup_event():
    await start_katago()


@app.get("/")
async def home():
    return {
        "status": "ok",
        "message": "KataGo WebSocket server running",
        "mode": MODE_NAME,
        "endpoints": {"monitor": "/monitor", "events": "/events", "status": "/status", "logs": "/logs", "katago_websocket": "/katago"},
    }


@app.get("/status")
async def status():
    STATE["pending_queries"] = len(pending)
    return {
        "ok": True,
        "uptime_seconds": round(time.time() - STATE["started_at"], 2),
        "state": STATE,
        "pending_queries": len(pending),
        "recent_logs": list(LOGS)[-50:],
    }


@app.get("/logs", response_class=PlainTextResponse)
async def logs(type: str = "all", q: str = ""):
    rows = list(LOGS)
    if type and type != "all":
        wanted = {t.strip() for t in type.split(",") if t.strip()}
        rows = [item for item in rows if item.get("type") in wanted]
    if q:
        q_lower = q.lower()
        rows = [item for item in rows if q_lower in item.get("message", "").lower()]
    return "\n".join(f"[{item['time']}] {item['type']}: {item['message']}" for item in rows)


@app.get("/events")
async def events(request: Request):
    queue = asyncio.Queue()
    EVENT_SUBSCRIBERS.add(queue)

    async def event_stream():
        try:
            for item in list(LOGS)[-50:]:
                yield f"data: {json.dumps(item)}\n\n"

            while True:
                if await request.is_disconnected():
                    break
                try:
                    item = await asyncio.wait_for(queue.get(), timeout=15)
                    yield f"data: {json.dumps(item)}\n\n"
                except asyncio.TimeoutError:
                    heartbeat = {"ts": now_ts(), "time": now_time(), "type": "heartbeat", "message": "alive", "state": dict(STATE)}
                    yield f"data: {json.dumps(heartbeat)}\n\n"
        finally:
            EVENT_SUBSCRIBERS.discard(queue)

    return StreamingResponse(event_stream(), media_type="text/event-stream")


MONITOR_HTML = """
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>KataGo Colab Monitor</title>
  <style>
    :root { color-scheme: dark; }
    body { font-family: system-ui, sans-serif; background: #0f1115; color: #e6e6e6; margin: 0; padding: 18px; }
    h1 { margin: 0 0 8px 0; font-size: 24px; }
    h2 { margin: 18px 0 10px; font-size: 18px; }
    .small { color: #9ca3af; font-size: 13px; margin-bottom: 14px; }
    .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 12px; margin-bottom: 18px; }
    .card { background: #171a21; border: 1px solid #2a2f3a; border-radius: 10px; padding: 12px; min-height: 70px; }
    .label { color: #9ca3af; font-size: 12px; margin-bottom: 6px; }
    .value { font-size: 20px; font-weight: 700; word-break: break-word; }
    .mono { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; font-size: 13px; }
    .ok { color: #61d394; } .bad { color: #ff6b6b; } .warn { color: #ffd166; } .muted { color: #9ca3af; }
    .toolbar { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; background: #171a21; border: 1px solid #2a2f3a; border-radius: 10px; padding: 10px; margin-bottom: 10px; }
    button, input { background: #0f1115; color: #e6e6e6; border: 1px solid #374151; border-radius: 8px; padding: 7px 9px; }
    button { cursor: pointer; }
    button.active { background: #2563eb; border-color: #60a5fa; }
    input { min-width: 220px; }
    #log { background: #05070a; border: 1px solid #2a2f3a; border-radius: 10px; padding: 12px; height: 55vh; overflow-y: auto; white-space: pre-wrap; font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; font-size: 13px; line-height: 1.45; }
    .event-system { color: #93c5fd; }
    .event-katago { color: #a7f3d0; }
    .event-tuning { color: #67e8f9; }
    .event-client { color: #fcd34d; }
    .event-query { color: #c4b5fd; }
    .event-result { color: #86efac; }
    .event-stale { color: #9ca3af; }
    .event-error { color: #fca5a5; }
    .event-warning { color: #fdba74; }
    .event-heartbeat { color: #6b7280; }
    code { background: #1f2937; padding: 2px 5px; border-radius: 4px; }
    a { color: #93c5fd; }
  </style>
</head>
<body>
  <h1>KataGo Colab Monitor</h1>
  <div class="small">KaTrain WebSocket path: <code>/katago</code> | Live endpoint: <code>/events</code> | Logs endpoint: <code>/logs?type=error,warning&q=text</code></div>

  <div class="grid">
    <div class="card"><div class="label">Mode</div><div id="mode" class="value">-</div></div>
    <div class="card"><div class="label">KataGo Ready</div><div id="katago_ready" class="value warn">checking...</div></div>
    <div class="card"><div class="label">Human Model</div><div id="human_model" class="value">-</div></div>
    <div class="card"><div class="label">Active Clients</div><div id="clients" class="value">0</div></div>
    <div class="card"><div class="label">Queries</div><div id="queries" class="value">0</div></div>
    <div class="card"><div class="label">Results</div><div id="results" class="value">0</div></div>
    <div class="card"><div class="label">Pending</div><div id="pending" class="value">0</div></div>
    <div class="card"><div class="label">Dropped</div><div id="dropped" class="value muted">0</div></div>
    <div class="card"><div class="label">Errors / Warnings</div><div class="value"><span id="errors" class="bad">0</span> / <span id="warnings" class="warn">0</span></div></div>
    <div class="card"><div class="label">Last Best Move</div><div id="best_move" class="value">-</div></div>
    <div class="card"><div class="label">Last Visits</div><div id="last_visits" class="value">-</div></div>
    <div class="card"><div class="label">Last Winrate</div><div id="last_winrate" class="value">-</div></div>
    <div class="card"><div class="label">Last Score Lead</div><div id="last_score" class="value">-</div></div>
    <div class="card"><div class="label">Last Query ID</div><div id="last_query" class="value mono">-</div></div>
    <div class="card"><div class="label">Last Result ID</div><div id="last_result" class="value mono">-</div></div>
    <div class="card"><div class="label">Last Error</div><div id="last_error" class="value bad mono">none</div></div>
  </div>

  <h2>Live Events</h2>
  <div class="toolbar">
    <button data-filter="all" class="active">All</button>
    <button data-filter="system">System</button>
    <button data-filter="katago">KataGo</button>
    <button data-filter="tuning">Tuning</button>
    <button data-filter="client">Client</button>
    <button data-filter="query">Query</button>
    <button data-filter="result">Result</button>
    <button data-filter="stale">Dropped</button>
    <button data-filter="warning">Warning</button>
    <button data-filter="error">Error</button>
    <button data-filter="heartbeat">Heartbeat</button>
    <input id="search" placeholder="Search logs...">
    <button id="pause">Pause</button>
    <button id="clear">Clear View</button>
  </div>
  <div id="log"></div>

  <script>
    const logEl = document.getElementById("log");
    const allEvents = [];
    const seenEvents = new Set();
    let currentFilter = "all";
    let paused = false;
    let sseFailures = 0;
    let pollingLogs = false;

    function setText(id, value) { document.getElementById(id).textContent = value ?? "-"; }
    function fmtFloat(x, digits=3) { return typeof x === "number" ? x.toFixed(digits) : "-"; }
    function fmtPct(x) { return typeof x === "number" ? (x * 100).toFixed(1) + "%" : "-"; }

    function updateState(state) {
      if (!state) return;
      setText("mode", state.mode || "-");
      setText("human_model", state.human_model_enabled ? "ON" : "OFF");
      const ready = document.getElementById("katago_ready");
      if (state.katago_ready) { ready.textContent = "YES"; ready.className = "value ok"; }
      else { ready.textContent = "NO"; ready.className = "value warn"; }
      setText("clients", state.websocket_clients_active);
      setText("queries", state.queries_received);
      setText("results", state.results_sent);
      setText("pending", state.pending_queries);
      setText("dropped", state.stale_results_dropped);
      setText("errors", state.errors);
      setText("warnings", state.warnings);
      setText("best_move", state.last_best_move || "-");
      setText("last_visits", state.last_visits ?? "-");
      setText("last_winrate", fmtPct(state.last_winrate));
      setText("last_score", fmtFloat(state.last_score_lead));
      setText("last_query", state.last_query_id || "-");
      setText("last_result", state.last_result_id || "-");
      setText("last_error", state.last_error || "none");
    }

    function passesFilter(item) {
      const q = document.getElementById("search").value.toLowerCase().trim();
      const typeOk = currentFilter === "all" || item.type === currentFilter;
      const textOk = !q || String(item.message || "").toLowerCase().includes(q) || String(item.type || "").toLowerCase().includes(q);
      return typeOk && textOk;
    }

    function renderLogs() {
      logEl.innerHTML = "";
      const visible = allEvents.filter(passesFilter).slice(-500);
      for (const item of visible) {
        const line = document.createElement("div");
        const type = item.type || "event";
        line.className = "event-" + type;
        line.textContent = `[${item.time}] ${type}: ${item.message}`;
        logEl.appendChild(line);
      }
      logEl.scrollTop = logEl.scrollHeight;
    }

    function addEvent(item) {
      updateState(item.state);
      const key = `${item.ts || ""}|${item.time || ""}|${item.type || ""}|${item.message || ""}`;
      if (seenEvents.has(key)) return;
      seenEvents.add(key);
      if (seenEvents.size > 2500) seenEvents.clear();
      allEvents.push(item);
      if (allEvents.length > 2000) allEvents.shift();
      if (!paused) renderLogs();
    }

    async function pollStatus() {
      try {
        const res = await fetch("/status");
        const data = await res.json();
        updateState(data.state);
        if (pollingLogs && Array.isArray(data.recent_logs)) {
          data.recent_logs.forEach(addEvent);
        }
      } catch (e) { console.log(e); }
    }

    document.querySelectorAll("button[data-filter]").forEach(btn => {
      btn.onclick = () => {
        document.querySelectorAll("button[data-filter]").forEach(b => b.classList.remove("active"));
        btn.classList.add("active");
        currentFilter = btn.dataset.filter;
        renderLogs();
      };
    });

    document.getElementById("search").oninput = renderLogs;
    document.getElementById("pause").onclick = () => {
      paused = !paused;
      document.getElementById("pause").textContent = paused ? "Resume" : "Pause";
      if (!paused) renderLogs();
    };
    document.getElementById("clear").onclick = () => { allEvents.length = 0; renderLogs(); };

    const forcePolling = new URLSearchParams(window.location.search).get("transport") === "poll";
    let source = null;
    if (forcePolling) {
      pollingLogs = true;
      addEvent({time: new Date().toLocaleTimeString(), type: "system", message: "Cloudflare Quick Tunnel monitor uses /status polling because Quick Tunnels do not support SSE.", state: null});
    } else {
      source = new EventSource("/events");
      source.onmessage = function(event) {
        try { addEvent(JSON.parse(event.data)); }
        catch (e) { console.log(e, event.data); }
      };
      source.onerror = function() {
        sseFailures += 1;
        addEvent({time: new Date().toLocaleTimeString(), type: "error", message: "Browser EventSource disconnected/reconnecting...", state: null});
        if (sseFailures >= 3 && !pollingLogs) {
          pollingLogs = true;
          source.close();
          addEvent({time: new Date().toLocaleTimeString(), type: "warning", message: "Live events unavailable; monitor switched to /status polling.", state: null});
        }
      };
    }

    pollStatus();
    setInterval(pollStatus, 5000);
  </script>
</body>
</html>
"""


@app.get("/monitor", response_class=HTMLResponse)
async def monitor():
    return HTMLResponse(MONITOR_HTML)


async def handle_katago_websocket(websocket: WebSocket):
    await websocket.accept()
    send_lock = asyncio.Lock()
    STATE["websocket_clients_total"] += 1
    STATE["websocket_clients_active"] += 1
    await emit("client", f"WebSocket client connected. Active clients: {STATE['websocket_clients_active']}")

    try:
        while True:
            message = await websocket.receive_text()
            try:
                query = json.loads(message)
            except Exception as exc:
                STATE["last_error"] = f"Bad JSON from websocket: {repr(exc)}"
                await emit("error", f"Bad JSON from websocket: {repr(exc)}")
                await emit("error", f"Message preview: {message[:500]}")
                continue

            query_id = query.get("id")
            terminate_id = query.get("terminateId")
            if query.get("action") == "terminate" and terminate_id:
                await terminate_katago_query(terminate_id, "client requested termination")
                continue

            if not query_id:
                query_id = str(uuid.uuid4())
                query["id"] = query_id

            await terminate_pending_query(query_id, "replaced by newer query with the same id")

            STATE["queries_received"] += 1
            STATE["last_query_id"] = query_id
            pending[query_id] = (websocket, send_lock)
            pending_expected_turns[query_id] = {int(turn) for turn in (query.get("analyzeTurns") or [])}
            pending_completed_turns[query_id] = set()
            STATE["pending_queries"] = len(pending)

            move_count = len(query.get("moves", []))
            max_visits = query.get("maxVisits")
            analyze_turns = query.get("analyzeTurns")
            override_settings = query.get("overrideSettings")
            await emit(
                "query",
                f"Received query id={query_id}, moves={move_count}, maxVisits={max_visits}, analyzeTurns={analyze_turns}, overrideSettings={override_settings}",
                {"query_id": query_id, "move_count": move_count, "max_visits": max_visits, "analyze_turns": analyze_turns, "override_settings": override_settings},
            )

            await write_to_katago(query)

    except WebSocketDisconnect:
        await emit("client", "WebSocket client disconnected.")
    except Exception as exc:
        STATE["last_error"] = repr(exc)
        await emit("error", f"WebSocket error: {repr(exc)}")
    finally:
        STATE["websocket_clients_active"] = max(0, STATE["websocket_clients_active"] - 1)
        dead_ids = [qid for qid, item in pending.items() if item[0] is websocket]
        for qid in dead_ids:
            await terminate_katago_query(qid, "websocket disconnected")
        STATE["pending_queries"] = len(pending)
        await emit("client", f"Client cleanup done. Active clients: {STATE['websocket_clients_active']}")


@app.websocket("/")
async def websocket_root(websocket: WebSocket):
    await handle_katago_websocket(websocket)


@app.websocket("/katago")
async def websocket_katago(websocket: WebSocket):
    await handle_katago_websocket(websocket)


nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=SERVER_PORT)
'''

    server_code = server_template
    server_code = server_code.replace("__MODE_NAME__", json.dumps(CONFIG_DATA["mode_name"]))
    server_code = server_code.replace("__KATAGO_BIN__", json.dumps(str(KATAGO_DIR / "katago")))
    server_code = server_code.replace("__MODEL__", json.dumps(str(MODEL_DIR / MODEL_FILENAME)))
    human_model_path = MODEL_DIR / HUMAN_MODEL_FILENAME
    server_code = server_code.replace("__HUMAN_MODEL__", json.dumps(str(human_model_path) if ENABLE_HUMAN_MODEL else ""))
    server_code = server_code.replace("__CONFIG__", json.dumps(str(KATAGO_DIR / "analysis.cfg")))
    server_code = server_code.replace("__SERVER_PORT__", str(SERVER_PORT))
    server_code = server_code.replace("__SUBPROCESS_LIMIT__", str(SUBPROCESS_BUFFER_MB * 1024 * 1024))
    server_code = server_code.replace("__LOG_LIMIT__", str(MONITOR_LOG_LIMIT))

    WS_SERVER_PATH.write_text(server_code)
    print(f"Wrote: {WS_SERVER_PATH}")


def start_ws_server():
    print("\n[8/12] Starting WebSocket server...")

    if WS_LOG_PATH.exists():
        WS_LOG_PATH.unlink()

    log = open(WS_LOG_PATH, "w")
    proc = subprocess.Popen(["python", str(WS_SERVER_PATH)], stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

    print("WebSocket server PID:", proc.pid)
    print("Waiting for KataGo startup...")
    started = False

    for i in range(KATAGO_STARTUP_TIMEOUT_SECONDS):
        time.sleep(1)
        if WS_LOG_PATH.exists():
            text = WS_LOG_PATH.read_text(errors="ignore")
            if "KataGo ready" in text or "Started, ready to begin handling requests" in text:
                started = True
                break
            if "Traceback" in text:
                print("\n--- Server log error ---")
                print(text[-8000:])
                raise RuntimeError("WebSocket server crashed.")
        if i % STARTUP_LOG_INTERVAL_SECONDS == 0:
            print(f"Still waiting... {i}s")
            if WS_LOG_PATH.exists():
                log_tail = WS_LOG_PATH.read_text(errors="ignore")[-STARTUP_LOG_TAIL_CHARS:]
                print(log_tail)

    print("\n--- WebSocket server log tail ---")
    print(WS_LOG_PATH.read_text(errors="ignore")[-5000:])

    if not started:
        raise RuntimeError(f"KataGo did not finish starting. Check {WS_LOG_PATH}")

    return proc


def test_local_http():
    print("\n[9/12] Testing local HTTP endpoints...")
    out = run(f"curl -s http://127.0.0.1:{SERVER_PORT}/", check=True)
    if "KataGo WebSocket server running" not in out:
        raise RuntimeError("Local HTTP test failed.")

    out = run(f"curl -s http://127.0.0.1:{SERVER_PORT}/status", check=True)
    if '"ok":true' not in out.replace(" ", ""):
        raise RuntimeError("Local status test failed.")


def test_local_websocket():
    print("\n[10/12] Testing local WebSocket...")
    visits = 10

    override_settings_line = ""
    if ENABLE_HUMAN_MODEL:
        override_settings_line = f',\n        "overrideSettings": {{"humanSLProfile": "{HUMAN_MODEL_TEST_PROFILE}"}}'

    test_code = f'''
import asyncio
import json
import websockets

async def test():
    uri = "ws://127.0.0.1:{SERVER_PORT}{KATAGO_WS_PATH}"
    query = {{
        "id": "local-ws-test",
        "moves": [["B", "Q16"], ["W", "D4"], ["B", "Q4"], ["W", "D16"]],
        "rules": "chinese",
        "komi": 7.5,
        "boardXSize": 19,
        "boardYSize": 19,
        "analyzeTurns": [4],
        "maxVisits": {visits},
        "includeOwnership": False,
        "includePolicy": False{override_settings_line}
    }}
    async with websockets.connect(uri, max_size={SUBPROCESS_BUFFER_MB} * 1024 * 1024) as ws:
        await ws.send(json.dumps(query))
        result = await ws.recv()
        print(result[:1000])

asyncio.run(test())
'''

    test_path = RUNTIME_BASE / "test_local_ws.py"
    test_path.write_text(test_code)
    out = run(f"python {test_path}", check=True)

    if '"moveInfos"' not in out and '"rootInfo"' not in out:
        raise RuntimeError("Local WebSocket test did not return KataGo analysis.")


def download_cloudflared():
    print("\n[11/12] Downloading cloudflared...")
    os.chdir(RUNTIME_BASE)

    download_file(CLOUDFLARED_URL, CLOUDFLARED_PATH, CLOUDFLARED_SHA256)
    run(f"chmod +x {CLOUDFLARED_PATH}", check=True)
    cloudflared_version = run(f"{CLOUDFLARED_PATH} --version", check=True)
    if CLOUDFLARED_VERSION not in cloudflared_version:
        raise RuntimeError(f"Expected cloudflared {CLOUDFLARED_VERSION}, got: {cloudflared_version}")


def make_public_urls(base_url, force_polling=False):
    base_url = base_url.rstrip("/")
    return {
        "base": base_url,
        "wss": base_url.replace("https://", "wss://").replace("http://", "ws://") + KATAGO_WS_PATH,
        "monitor": base_url + "/monitor" + ("?transport=poll" if force_polling else ""),
        "status": base_url + "/status",
        "logs": base_url + "/logs",
    }


def print_tunnel_success(provider_name, proc, urls):
    print("\n" + "=" * 90)
    print("COPY THIS INTO KATRAIN -> SETTINGS -> REMOTE ENGINE -> KATAGO WEBSOCKET URL")
    print()
    print(urls["wss"])
    print()
    print("=" * 90)

    print("\nTunnel provider:", provider_name)
    print("\nLIVE BROWSER MONITOR:")
    print(urls["monitor"])

    print("\nOther useful links:")
    print("Status:", urls["status"])
    print("Logs:  ", urls["logs"])
    print("Filtered logs examples:")
    print("Errors:", urls["logs"] + "?type=error")
    print("Tuning:", urls["logs"] + "?type=tuning")
    print("Search:", urls["logs"] + "?q=humanSLProfile")

    print("\nKaTrain steps:")
    print("1. Open KaTrain settings.")
    print("2. Select Remote engine tab.")
    print("3. Paste the WebSocket URL printed above.")
    print("4. Click Update Settings.")
    print("5. Fully close KaTrain.")
    print("6. Open KaTrain again.")
    print("7. Open /monitor in browser to watch live events and use log filters.")

    return proc, urls["wss"], urls["monitor"]


def start_cloudflare_tunnel():
    print("\nStarting Cloudflare Quick Tunnel...")

    if CLOUDFLARE_LOG_PATH.exists():
        CLOUDFLARE_LOG_PATH.unlink()

    log = open(CLOUDFLARE_LOG_PATH, "w")
    proc = subprocess.Popen([str(CLOUDFLARED_PATH), "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}"], stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

    print("cloudflared PID:", proc.pid)
    print("Waiting for trycloudflare URL...")
    url = None

    for i in range(CLOUDFLARE_TIMEOUT_SECONDS):
        time.sleep(1)
        if CLOUDFLARE_LOG_PATH.exists():
            text = CLOUDFLARE_LOG_PATH.read_text(errors="ignore")
            match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", text)
            if match:
                url = match.group(0)
                break
        if i % 10 == 0:
            print(f"Still waiting for tunnel URL... {i}s")

    print("\n--- Cloudflare log tail ---")
    print(CLOUDFLARE_LOG_PATH.read_text(errors="ignore")[-5000:])

    if not url:
        raise RuntimeError(f"Could not find Cloudflare URL. Check {CLOUDFLARE_LOG_PATH}")

    return print_tunnel_success("cloudflare", proc, make_public_urls(url, force_polling=True))


def ssh_common_options():
    return [
        "-o", "StrictHostKeyChecking=no",
        "-o", "UserKnownHostsFile=/dev/null",
        "-o", f"ServerAliveInterval={SSH_SERVER_ALIVE_INTERVAL_SECONDS}",
        "-o", f"ConnectTimeout={SSH_CONNECT_TIMEOUT_SECONDS}",
        "-o", "ExitOnForwardFailure=yes",
    ]


def read_log_tail(path, chars=5000):
    if not path.exists():
        return ""
    return path.read_text(errors="ignore")[-chars:]


def start_pinggy_tunnel():
    print("\nStarting Pinggy free SSH tunnel...")

    if PINGGY_LOG_PATH.exists():
        PINGGY_LOG_PATH.unlink()

    cmd = [
        "ssh",
        *ssh_common_options(),
        "-p", "443",
        "-L", f"{PINGGY_DEBUG_PORT}:localhost:{PINGGY_DEBUG_PORT}",
        "-R", f"0:127.0.0.1:{SERVER_PORT}",
        "free.pinggy.io",
    ]
    log = open(PINGGY_LOG_PATH, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

    print("Pinggy SSH PID:", proc.pid)
    print("Waiting for Pinggy URL...")
    url = None

    for i in range(PINGGY_TIMEOUT_SECONDS):
        time.sleep(1)
        text = read_log_tail(PINGGY_LOG_PATH, 12000)
        match = re.search(r"https://[a-zA-Z0-9\-.]+pinggy(?:-free)?\.link", text)
        if match:
            url = match.group(0)
            break

        code, out = command_output(f"curl -s http://127.0.0.1:{PINGGY_DEBUG_PORT}/urls")
        if code == 0:
            match = re.search(r"https://[a-zA-Z0-9\-.]+pinggy(?:-free)?\.link", out)
            if match:
                url = match.group(0)
                break

        if proc.poll() is not None:
            raise RuntimeError(f"Pinggy tunnel exited early. Check {PINGGY_LOG_PATH}")
        if i % 10 == 0:
            print(f"Still waiting for Pinggy URL... {i}s")

    print("\n--- Pinggy log tail ---")
    print(read_log_tail(PINGGY_LOG_PATH))

    if not url:
        raise RuntimeError(f"Could not find Pinggy URL. Check {PINGGY_LOG_PATH}")

    return print_tunnel_success("pinggy", proc, make_public_urls(url))


def start_localhostrun_tunnel():
    print("\nStarting localhost.run free SSH tunnel...")

    if LOCALHOSTRUN_LOG_PATH.exists():
        LOCALHOSTRUN_LOG_PATH.unlink()

    cmd = [
        "ssh",
        *ssh_common_options(),
        "-R", f"80:127.0.0.1:{SERVER_PORT}",
        "nokey@localhost.run",
    ]
    log = open(LOCALHOSTRUN_LOG_PATH, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)

    print("localhost.run SSH PID:", proc.pid)
    print("Waiting for localhost.run URL...")
    url = None

    for i in range(LOCALHOSTRUN_TIMEOUT_SECONDS):
        time.sleep(1)
        text = read_log_tail(LOCALHOSTRUN_LOG_PATH, 12000)
        matches = re.findall(r"https://[a-zA-Z0-9\-._~:/?#\[\]@!$&'()*+,;=%]+", text)
        for candidate in matches:
            if "localhost.run" in candidate or "lhr.life" in candidate or "lhr.rocks" in candidate:
                url = candidate.rstrip(".,)")
                break
        if url:
            break

        if proc.poll() is not None:
            raise RuntimeError(f"localhost.run tunnel exited early. Check {LOCALHOSTRUN_LOG_PATH}")
        if i % 10 == 0:
            print(f"Still waiting for localhost.run URL... {i}s")

    print("\n--- localhost.run log tail ---")
    print(read_log_tail(LOCALHOSTRUN_LOG_PATH))

    if not url:
        raise RuntimeError(f"Could not find localhost.run URL. Check {LOCALHOSTRUN_LOG_PATH}")

    return print_tunnel_success("localhostrun", proc, make_public_urls(url))


def selected_tunnel_providers():
    provider = TUNNEL_PROVIDER.lower().strip()
    allowed = {"auto", "cloudflare", "pinggy", "localhostrun", "none"}
    if provider not in allowed:
        raise ValueError(f"TUNNEL_PROVIDER must be one of {sorted(allowed)}")
    if provider == "none":
        return []
    if provider != "auto":
        return [provider]

    providers = []
    for item in TUNNEL_FALLBACK_ORDER:
        item = str(item).lower().strip()
        if item not in {"cloudflare", "pinggy", "localhostrun"}:
            raise ValueError(f"Unknown tunnel provider in TUNNEL_FALLBACK_ORDER: {item}")
        if item not in providers:
            providers.append(item)
    return providers


def start_public_tunnel():
    print("\n[12/12] Starting public tunnel...")
    providers = selected_tunnel_providers()
    if not providers:
        local_ws = f"ws://127.0.0.1:{SERVER_PORT}{KATAGO_WS_PATH}"
        local_monitor = f"http://127.0.0.1:{SERVER_PORT}/monitor"
        print("Public tunnel disabled by TUNNEL_PROVIDER='none'.")
        print("Local WebSocket URL:", local_ws)
        print("Local Monitor URL:", local_monitor)
        return None, local_ws, local_monitor

    starters = {
        "cloudflare": start_cloudflare_tunnel,
        "pinggy": start_pinggy_tunnel,
        "localhostrun": start_localhostrun_tunnel,
    }
    errors = []
    for provider in providers:
        try:
            return starters[provider]()
        except Exception as exc:
            errors.append(f"{provider}: {exc}")
            print(f"\nTunnel provider failed ({provider}): {exc}")
            print("Trying next tunnel provider..." if provider != providers[-1] else "No tunnel providers left.")

    raise RuntimeError("All tunnel providers failed:\n" + "\n".join(errors))


# ============================================================
# Run everything
# ============================================================

print("Selected mode:", CONFIG_DATA["mode_name"])

if STOP_OLD_PROCESSES:
    stop_old_processes()
else:
    print("\n[1/12] Skipping old process cleanup because STOP_OLD_PROCESSES=False")

install_dependencies()
print_system_info()
download_katago()
download_model()
write_analysis_config()
write_ws_server()

ws_proc = start_ws_server()
test_local_http()
test_local_websocket()

tunnel_providers = selected_tunnel_providers()
if "cloudflare" in tunnel_providers:
    download_cloudflared()
else:
    print("\n[11/12] Skipping cloudflared download because Cloudflare is not selected.")

tunnel_proc, final_wss_url, final_monitor_url = start_public_tunnel()

print("\nSETUP COMPLETE")
print("\nFinal KaTrain WebSocket URL:")
print(final_wss_url)
print("\nLive Monitor URL:")
print(final_monitor_url)


## Optional: Stop the Server Before Closing Colab

Run this cell when you are done, or before restarting the setup cell manually. It stops KataGo, the WebSocket server, and any public tunnel processes started by this notebook.



In [ ]:
# @title Stop KataGo Remote Engine
# Stop the server before closing

import os, signal, subprocess, time

patterns = [
    "katago_ws_server.py",
    "katago analysis",
    "cloudflared tunnel",
    "free.pinggy.io",
    "localhost.run",
]

for line in subprocess.check_output(["ps", "-eo", "pid=,args="], text=True).splitlines():
    line = line.strip()
    if not line:
        continue

    parts = line.split(None, 1)
    if len(parts) < 2:
        continue

    pid = int(parts[0])
    cmd = parts[1]

    if any(p in cmd for p in patterns):
        print("Stopping:", pid, cmd)
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass

time.sleep(2)
print("Done.")

